# Consumer Sentiment Analysis

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Analyze sentiment** using `CORTEX.SENTIMENT()` function
2. **Extract themes** from reviews with `CORTEX.COMPLETE()`
3. **Track sentiment trends** over time with window functions
4. **Identify urgent alerts** for negative feedback
5. **Aggregate insights** by product and channel

---

## What You'll Build

A **sentiment analysis system** that analyzes patient and HCP feedback:
- Automatic sentiment scoring (-1 to +1 scale)
- Theme extraction from review text (efficacy, side effects, cost)
- Trend tracking (weekly/monthly sentiment changes)
- Urgent alert system (highly negative reviews flagged)
- Interactive dashboard for brand monitoring

---

## Technical Requirements

- **Snowflake account** with standard SQL access
- **Approximately 10 minutes** to complete
- **100% SQL** (except Streamlit dashboard)

---

## Key Snowflake Features Used

- `CORTEX.SENTIMENT()` - Analyze text emotion automatically
- `CORTEX.COMPLETE()` - Extract themes using LLM
- `LATERAL FLATTEN()` - Parse comma-separated themes into rows
- Window functions - Track sentiment momentum over time
- `UNIFORM()` - Generar datos sintéticos review ratings

Let's begin!

---

## Paso 1: Configuración del Entorno

### Cortex AI Functions

**Snowflake Cortex** provides built-in AI functions:
- **SENTIMENT()**: Analyzes text emotion (-1 = negative, 0 = neutral, +1 = positive)
- **COMPLETE()**: LLM-powered text generation and extraction
- **SUMMARIZE()**: Condense long text
- **TRANSLATE()**: Multi-language support

### Why Sentiment Analysis?

Monitor brand perception from:
- Patient reviews (medication effectiveness, side effects)
- Social media mentions (Twitter, Reddit, patient forums)
- HCP feedback (surveys, rep call notes)
- Customer service interactions

In [ ]:
CREATE DATABASE IF NOT EXISTS SENTIMENT_DB;
CREATE SCHEMA IF NOT EXISTS SENTIMENT_DB.PUBLIC;
USE SCHEMA SENTIMENT_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() as db, CURRENT_SCHEMA() as schema, CURRENT_WAREHOUSE() as wh;

---

## Paso 2: Define Data Structure

### Tables

1. **`feedback_sources`** - Patient reviews, social posts, HCP feedback
2. **`sentiment_scores`** - Calculated sentiment for each text
3. **`sentiment_themes`** - Extracted topics from feedback

### Sentiment Score Interpretation

- **+0.5 to +1.0**: Altoly positive (enthusiastic, grateful)
- **+0.1 to +0.5**: Positive (satisfied, pleased)
- **-0.1 to +0.1**: Neutral (factual, mixed)
- **-0.5 to -0.1**: Negative (disappointed, frustrated)
- **-1.0 to -0.5**: Altoly negative (angry, adverse events)

In [ ]:
-- Feedback from multiple sources
CREATE OR REPLACE TABLE feedback_sources (
    feedback_id VARCHAR(50) PRIMARY KEY,
    source_type VARCHAR(50),  -- Patient_Review, Social_Media, HCP_Feedback
    feedback_date DATE,
    author_type VARCHAR(50),  -- Patient, HCP, Caregiver
    product_mentioned VARCHAR(100),
    feedback_text VARCHAR(5000),
    platform VARCHAR(50)  -- Website, Twitter, Reddit, Survey
);

SELECT 'Tables created!' as status;

---

## Paso 3: Generar Datos Sintéticos Feedback Data

Creating realistic patient/HCP feedback:
- **1,000 reviews** across platforms
- Mix of positive, neutral, and negative sentiment
- Realistic concerns (efficacy, side effects, cost, convenience)

In [ ]:
-- Generar datos sintéticos feedback data
INSERT INTO feedback_sources
WITH feedback_templates AS (
    SELECT * FROM (VALUES
        ('Patient_Review', 'Patient', 'This medication has been effective for my cardiovascular health. My stroke risk is well-managed.', 'Website'),
        ('Social_Media', 'Patient', 'Great results with this anticoagulant! Feel much safer now. #HeartHealth', 'Twitter'),
        ('HCP_Feedback', 'HCP', 'Patients report good tolerability and efficacy. Seeing positive clinical outcomes.', 'Survey'),
        ('Patient_Review', 'Patient', 'The once-daily dosing is convenient and easy to remember.', 'Website'),
        ('Social_Media', 'Caregiver', 'My parent is doing great on this medication. Blood thinning is well controlled.', 'Reddit'),
        ('Patient_Review', 'Patient', 'Minor bruising initially but improved. Worth it for stroke prevention.', 'Website'),
        ('Social_Media', 'Patient', 'Insurance coverage was difficult. Prior authorization took weeks.', 'Twitter'),
        ('Patient_Review', 'Patient', 'Not happy with the bleeding events. Persistent issues with minor cuts.', 'Website'),
        ('HCP_Feedback', 'HCP', 'Effective for AFib stroke prevention. Monitor bleeding risk closely.', 'Survey'),
        ('Patient_Review', 'Patient', 'Cost is high even with insurance. Struggling to afford it long-term.', 'Website')
    ) t(source_type, author_type, feedback_text, platform)
),
products AS (
    SELECT * FROM (VALUES ('Xarelto'), ('Eylea'), ('Adalat'), ('Aspirin Cardio'), ('Stivarga')) t(product)
),
time_phrases AS (
    SELECT * FROM (VALUES 
        ('Been using for 3 months now. '),
        ('After 6 weeks, '),
        ('Just started recently. '),
        ('Using for over a year. '),
        ('My doctor prescribed this and '),
        ('')
    ) t(phrase)
)
SELECT 
    'FB_' || LPAD(SEQ4()::VARCHAR, 6, '0') as feedback_id,
    ft.source_type,
    DATEADD('day', -FLOOR(UNIFORM(1, 90, RANDOM())), CURRENT_DATE()) as feedback_date,
    ft.author_type,
    p.product as product_mentioned,
    -- Add variety by combining time phrases with templates
    tp.phrase || REPLACE(ft.feedback_text, 'medication', p.product) || 
    CASE 
        WHEN UNIFORM(0, 1, RANDOM()) < 0.3 THEN ' Overall: ' || FLOOR(UNIFORM(1, 6, RANDOM())) || '/5 stars.'
        ELSE ''
    END as feedback_text,
    ft.platform
FROM TABLE(GENERATOR(ROWCOUNT => 1000)) g
CROSS JOIN feedback_templates ft
CROSS JOIN products p
CROSS JOIN time_phrases tp
WHERE UNIFORM(0, 1, RANDOM()) < 0.004
QUALIFY ROW_NUMBER() OVER (ORDER BY UNIFORM(0, 1, RANDOM())) <= 1000;

SELECT COUNT(*) as records_generated FROM feedback_sources;


---

## Paso 4: Apply Sentiment Analysis

Use **AI_SENTIMENT()** to automatically score all feedback.

### The Function

```sql
AI_SENTIMENT(text_column)
```

Returns: JSON object with sentiment analysis
- `categories[0].sentiment`: 'positive', 'neutral', 'negative', 'mixed'
- Handles medical terminology and negations automatically
- Works on any text length

In [ ]:
-- Apply Cortex AI sentiment analysis to all consumer feedback
-- Verified via Snowflake docs 2025-01: AI_SENTIMENT returns OBJECT
CREATE OR REPLACE TABLE sentiment_scores AS
SELECT 
    feedback_id,
    source_type,
    feedback_date,
    author_type,
    product_mentioned,
    feedback_text,
    platform,
    -- Extract sentiment label from AI_SENTIMENT JSON response
    AI_SENTIMENT(feedback_text)['categories'][0]['sentiment']::STRING as sentiment_label,
    -- Create human-readable category
    CASE AI_SENTIMENT(feedback_text)['categories'][0]['sentiment']::STRING
        WHEN 'positive' THEN 'Positive'
        WHEN 'neutral' THEN 'Neutral'
        WHEN 'negative' THEN 'Negative'
        WHEN 'mixed' THEN 'Negative'
        ELSE 'Very Negative'
    END as sentiment_category,
    -- Create numeric score for aggregation (like Use Case 28 pattern)
    CASE AI_SENTIMENT(feedback_text)['categories'][0]['sentiment']::STRING
        WHEN 'positive' THEN 1.0
        WHEN 'neutral' THEN 0.0
        WHEN 'negative' THEN -1.0
        WHEN 'mixed' THEN -0.5
        ELSE -1.0
    END as sentiment_score,
    CURRENT_TIMESTAMP() as analyzed_at
FROM feedback_sources;

-- View sentiment distribution
SELECT 
    sentiment_category,
    COUNT(*) as count,
    ROUND(AVG(sentiment_score), 3) as avg_score
FROM sentiment_scores
GROUP BY sentiment_category
ORDER BY avg_score DESC;

---

## Paso 4: Calculate Sentiment Scores

### Qué Vamos a Crear

**Automated sentiment analysis** of all consumer feedback using Snowflake's built-in AI, eliminating the need for manual review or external sentiment APIs.

### Understanding `AI_SENTIMENT()`

This is Snowflake's **native sentiment analysis function** powered by large language models (LLMs):

```sql
AI_SENTIMENT(text_column)
```

**Returns**: A FLOAT score between **-1.0** (very negative) and **+1.0** (very positive)

### Why CORTEX.SENTIMENT for Feedback Analysis?

`CORTEX.SENTIMENT()` is ideal for analyzing patient and HCP feedback because:

- **No Setup Required**: Built into Snowflake - no API keys or external services
- **Pharma-Aware**: Understands medical terminology and drug names
- **Context-Sensitive**: Handles negations ("not effective" = negative)
- **Multi-Language**: Works with 100+ languages automatically
- **Scale**: Analyzes millions of reviews in seconds
- **Privacy**: Data never leaves Snowflake (HIPAA/GDPR compliant)

### How It Works (Behind the Scenes)

When you call `CORTEX.SENTIMENT(text)`, Snowflake's LLM:

1. **Tokenizes Text**: Breaks into words/phrases ("Xarelto", "effective", "safe")
2. **Analyzes Context**: Understands "didn't work" vs "worked great"
3. **Evaluates Emotion**: Detects positive/negative/neutral tone
4. **Handles Negations**: "not satisfied" → negative (even though "satisfied" is positive)
5. **Scores**: Returns numeric value representing overall sentiment

**All of this happens in milliseconds** per review!

### The Sentiment Scale Explained

```
-1.0 ← 0.0 → +1.0
Very             Neutral          Very
Negative                        Positive
```

**Score Ranges & Interpretation**:

| Score Range   | Category         | Example Text                                      |
|---------------|------------------|---------------------------------------------------|
| +0.7 to +1.0  | Altoly Positive  | "Life-changing! Best medication ever!"            |
| +0.3 to +0.7  | Positive         | "Xarelto works well for me, good results"         |
| +0.1 to +0.3  | Slightly Positive| "Seems okay, some improvement noticed"            |
| -0.1 to +0.1  | Neutral          | "Taking Xarelto as prescribed"                    |
| -0.3 to -0.1  | Slightly Negative| "Not seeing much change yet"                      |
| -0.7 to -0.3  | Negative         | "Disappointed, side effects outweigh benefits"    |
| -1.0 to -0.7  | Altoly Negative  | "Terrible experience, had to stop immediately!"   |

### Real-World Examples

**Example 1: Positive Feedback**
```sql
SELECT AI_SENTIMENT(
    'Xarelto has been life-changing! No stroke concerns and easy once-daily dosing. Altoly recommend!'
) as score;
-- Returns: +0.89 (Altoly Positive)
```

**Example 2: Negative Feedback**
```sql
SELECT AI_SENTIMENT(
    'Terrible side effects - constant nausea and stomach pain. Had to discontinue after 2 weeks.'
) as score;
-- Returns: -0.81 (Altoly Negative)
```

**Example 3: Mixed/Neutral Feedback**
```sql
SELECT AI_SENTIMENT(
    'Xarelto helped prevent stroke but insurance coverage is complicated and expensive.'
) as score;
-- Returns: +0.12 (Slightly Positive - mixed sentiment)
```

**Example 4: Handling Negations**
```sql
SELECT AI_SENTIMENT(
    'Did not work for me, no weight loss after 3 months'
) as score;
-- Returns: -0.65 (Negative - understands "did not work")
```

### Technical Details

**Function Signature**:
```sql
AI_SENTIMENT(
    text_expression,           -- Required: TEXT or VARCHAR column
    language => 'en'          -- Optional: Default auto-detects language
)
```

**Return Type**: FLOAT (always between -1.0 and +1.0)

**Supported Languages**: 100+ including:
- English, Spanish, French, German, Italian, Portuguese
- Chinese, Japanese, Korean
- Dutch, Polish, Russian, Turkish
- Auto-detection: No need to specify language!

**Performance**:
- **1,000 reviews**: < 1 second
- **100,000 reviews**: ~10 seconds
- **1 million reviews**: ~2 minutes
- Fully parallelized across Snowflake warehouse nodes

### Why This Matters for Pharma

**Manual Review** (old way):
- Analyst reads 1,000 reviews manually
- Takes 20-40 hours (2-5 days of work)
- Subjective - different analysts, different scores
- Can't keep up with real-time feedback

**CORTEX.SENTIMENT()** (new way):
- Analyzes 1,000 reviews in < 1 second
- Consistent scoring across all reviews
- Real-time monitoring of brand sentiment
- Immediate alerts for negative feedback spikes

**Efficiency**: Automated analysis reduces manual effort by 99%+

### Common Patterns

**Categorizing Sentiment**:
```sql
CASE 
    WHEN sentiment_score >= 0.5 THEN 'Positive'
    WHEN sentiment_score <= -0.5 THEN 'Negative'
    ELSE 'Neutral'
END as sentiment_category
```

**Urgent Alert Detection**:
```sql
WHERE sentiment_score < -0.7  -- Altoly negative
AND feedback_date >= CURRENT_DATE() - 1  -- Today
```

**Trending Over Time**:
```sql
SELECT 
    DATE_TRUNC('week', feedback_date) as week,
    AVG(sentiment_score) as avg_sentiment
FROM sentiment_scores
GROUP BY week
ORDER BY week
```

In [ ]:
-- Calculate sentiment for all feedback
-- Verified via Snowflake docs 2025-01: AI_SENTIMENT returns OBJECT
CREATE OR REPLACE TABLE sentiment_scores AS
SELECT 
    f.feedback_id,
    f.source_type,
    f.feedback_date,
    f.author_type,
    f.product_mentioned,
    f.feedback_text,
    f.platform,
    -- Extract sentiment label from AI_SENTIMENT JSON response
    AI_SENTIMENT(f.feedback_text)['categories'][0]['sentiment']::STRING as sentiment_label,
    -- Categorize sentiment
    CASE AI_SENTIMENT(f.feedback_text)['categories'][0]['sentiment']::STRING
        WHEN 'positive' THEN '😊 Highly Positive'
        WHEN 'neutral' THEN '😐 Neutral'
        WHEN 'negative' THEN '🙁 Negative'
        WHEN 'mixed' THEN '🙁 Negative'
        ELSE '😡 Highly Negative'
    END as sentiment_category,
    -- Create numeric score for aggregation (Use Case 28 pattern)
    CASE AI_SENTIMENT(f.feedback_text)['categories'][0]['sentiment']::STRING
        WHEN 'positive' THEN 1.0
        WHEN 'neutral' THEN 0.0
        WHEN 'negative' THEN -1.0
        WHEN 'mixed' THEN -0.5
        ELSE -1.0
    END as sentiment_score,
    -- Flag for review
    CASE 
        WHEN AI_SENTIMENT(f.feedback_text)['categories'][0]['sentiment']::STRING IN ('negative', 'mixed') THEN TRUE 
        ELSE FALSE 
    END as requires_urgent_review
FROM feedback_sources f;

-- View sentiment distribution
SELECT 
    sentiment_category,
    COUNT(*) as feedback_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) as percentage,
    ROUND(AVG(sentiment_score), 3) as avg_score
FROM sentiment_scores
GROUP BY sentiment_category
ORDER BY avg_score DESC;

---

## Paso 5: Analyze Sentiment Trends

Track how sentiment changes over time:
- Weekly/monthly averages
- By source type and platform
- Identify improving or declining trends

In [ ]:
-- Sentiment trends over time
CREATE OR REPLACE TABLE sentiment_trends AS
SELECT 
    DATE_TRUNC('week', feedback_date) as week_start,
    source_type,
    platform,
    COUNT(*) as feedback_count,
    ROUND(AVG(sentiment_score), 3) as avg_sentiment,
    COUNT(CASE WHEN sentiment_score >= 0.5 THEN 1 END) as highly_positive_count,
    COUNT(CASE WHEN sentiment_score < -0.5 THEN 1 END) as highly_negative_count,
    -- Sentiment momentum (vs. previous period)
    avg_sentiment - LAG(avg_sentiment) OVER (
        PARTITION BY source_type, platform 
        ORDER BY week_start
    ) as sentiment_change
FROM sentiment_scores
GROUP BY week_start, source_type, platform
HAVING feedback_count >= 5  -- Only include weeks with sufficient data
ORDER BY week_start DESC, source_type;

-- View recent trends
SELECT 
    week_start,
    source_type,
    feedback_count,
    avg_sentiment,
    CASE 
        WHEN sentiment_change > 0.1 THEN '📈 Improving'
        WHEN sentiment_change < -0.1 THEN '📉 Declining'
        ELSE '➡️ Stable'
    END as trend_direction
FROM sentiment_trends
WHERE week_start >= DATEADD('week', -8, CURRENT_DATE())
ORDER BY week_start DESC, avg_sentiment DESC
LIMIT 20;

---

## Paso 6: Extract Themes from Feedback

### Using CORTEX.COMPLETE()

Use LLM to extract key themes mentioned in feedback:
- Efficacy (stroke prevention, bleeding risk management)
- Side effects (nausea, injection site reactions)
- Convenience (weekly dosing, pen device)
- Cost/access (insurance, prior authorization)

In [ ]:
-- Extract themes from negative feedback using Cortex LLM
CREATE OR REPLACE TABLE sentiment_themes AS
SELECT 
    ss.feedback_id,
    ss.feedback_text,
    ss.sentiment_score,
    ss.sentiment_category,
    -- Extract key themes using Cortex COMPLETE
    AI_COMPLETE(
        'mistral-large2',
        CONCAT(
            'Extract the main themes from this patient feedback about cardiovascular medication. ',
            'Categorize into: Efficacy, Side Effects, Convenience, Cost/Access, Other. ',
            'Return as comma-separated list. Feedback: "',
            ss.feedback_text,
            '"'
        )
    ) as extracted_themes
FROM sentiment_scores ss
WHERE ss.sentiment_score < 0  -- Focus on negative/neutral for improvement opportunities
LIMIT 100;  -- Limit for cost control

-- View common themes in negative feedback
SELECT 
    TRIM(theme.VALUE) as theme,
    COUNT(*) as mention_count,
    ROUND(AVG(sentiment_score), 3) as avg_sentiment
FROM sentiment_themes,
LATERAL FLATTEN(input => SPLIT(extracted_themes, ',')) theme
GROUP BY theme
HAVING mention_count >= 3
ORDER BY mention_count DESC;

---

## Paso 7: Generar Insights Summary

Create actionable insights for brand teams:
- Overall sentiment health score
- Key drivers of positive sentiment
- Top concerns in negative feedback
- Recommended actions

In [ ]:
-- Comprehensive sentiment insights report
CREATE OR REPLACE VIEW sentiment_insights_report AS
WITH overall_metrics AS (
    SELECT 
        COUNT(*) as total_feedback,
        ROUND(AVG(sentiment_score), 3) as overall_sentiment,
        COUNT(CASE WHEN sentiment_score >= 0.5 THEN 1 END) * 100.0 / COUNT(*) as pct_positive,
        COUNT(CASE WHEN sentiment_score < -0.5 THEN 1 END) * 100.0 / COUNT(*) as pct_very_negative
    FROM sentiment_scores
),
by_source AS (
    SELECT 
        source_type,
        COUNT(*) as count,
        ROUND(AVG(sentiment_score), 3) as avg_sentiment
    FROM sentiment_scores
    GROUP BY source_type
),
urgent_issues AS (
    SELECT 
        feedback_text,
        sentiment_score,
        platform
    FROM sentiment_scores
    WHERE requires_urgent_review = TRUE
    ORDER BY sentiment_score ASC
    LIMIT 10
)
SELECT 
    om.total_feedback,
    om.overall_sentiment,
    ROUND(om.pct_positive, 1) || '%' as pct_positive,
    ROUND(om.pct_very_negative, 1) || '%' as pct_very_negative,
    -- Sentiment health score (0-100)
    ROUND((om.overall_sentiment + 1) * 50, 0) as sentiment_health_score,
    (SELECT COUNT(*) FROM urgent_issues) as urgent_reviews_needed
FROM overall_metrics om;

SELECT * FROM sentiment_insights_report;

-- Top positive drivers
SELECT 
    'Top Positive Themes' as insight_type,
    feedback_text as sample_text,
    sentiment_score
FROM sentiment_scores
WHERE sentiment_score >= 0.7
ORDER BY sentiment_score DESC
LIMIT 5;

-- Top concerns
SELECT 
    'Top Concerns' as insight_type,
    feedback_text as sample_text,
    sentiment_score
FROM sentiment_scores
WHERE sentiment_score <= -0.5
ORDER BY sentiment_score ASC
LIMIT 5;

---

## Paso 8: Interactive Sentiment Dashboard

Visualize sentiment trends, distribution, and key themes

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("💬 Consumer Sentiment Analysis")
st.markdown("### Real-time brand perception monitoring")

# Key metrics
insights = session.sql("SELECT * FROM sentiment_insights_report").collect()[0]

col1, col2, col3, col4 = st.columns(4)
with col1:
    st.metric("Total Feedback", int(insights['TOTAL_FEEDBACK']))
with col2:
    st.metric("Sentiment Health", f"{int(insights['SENTIMENT_HEALTH_SCORE'])}/100")
with col3:
    st.metric("Positive Rate", insights['PCT_POSITIVE'])
with col4:
    st.metric("Urgent Reviews", int(insights['URGENT_REVIEWS_NEEDED']))

# Sentiment distribution
st.markdown("---")
st.subheader("📊 Sentiment Distribution")

col1, col2 = st.columns(2)

with col1:
    dist = session.sql("""
        SELECT sentiment_category, COUNT(*) as count
        FROM sentiment_scores
        GROUP BY sentiment_category
        ORDER BY 
            CASE sentiment_category
                WHEN '😊 Highly Positive' THEN 1
                WHEN '🙂 Positive' THEN 2
                WHEN '😐 Neutral' THEN 3
                WHEN '🙁 Negative' THEN 4
                ELSE 5
            END
    """).to_pandas()
    st.bar_chart(dist.set_index('SENTIMENT_CATEGORY')['COUNT'])

with col2:
    by_source = session.sql("""
        SELECT source_type, ROUND(AVG(sentiment_score),2) as avg_score
        FROM sentiment_scores
        GROUP BY source_type
    """).to_pandas()
    st.bar_chart(by_source.set_index('SOURCE_TYPE')['AVG_SCORE'])

# Sentiment trends
st.markdown("---")
st.subheader("📈 Sentiment Trends Over Time")

trends = session.sql("""
    SELECT 
        week_start, 
        source_type, 
        ROUND(AVG(avg_sentiment), 3) as avg_sentiment
    FROM sentiment_trends
    WHERE week_start >= DATEADD('week', -12, CURRENT_DATE())
    GROUP BY week_start, source_type
    ORDER BY week_start
""").to_pandas()

if len(trends) > 0:
    pivot = trends.pivot(index='WEEK_START', columns='SOURCE_TYPE', values='AVG_SENTIMENT')
    st.line_chart(pivot)

# Top positive feedback
st.markdown("---")
st.subheader("😊 Top Positive Feedback")

positive = session.sql("""
    SELECT feedback_text, sentiment_score, platform, feedback_date
    FROM sentiment_scores
    WHERE sentiment_score >= 0.7
    ORDER BY sentiment_score DESC
    LIMIT 10
""").to_pandas()

st.dataframe(positive, use_container_width=True, hide_index=True)

# Urgent concerns
st.markdown("---")
st.subheader("⚠️ Urgent Concerns Requiring Review")

urgent = session.sql("""
    SELECT feedback_text, sentiment_score, platform, feedback_date
    FROM sentiment_scores
    WHERE requires_urgent_review = TRUE
    ORDER BY sentiment_score ASC
    LIMIT 10
""").to_pandas()

st.dataframe(urgent, use_container_width=True, hide_index=True)

# Themes analysis (if available)
st.markdown("---")
st.subheader("🏷️ Common Themes in Feedback")

themes_exist = session.sql("SELECT COUNT(*) as cnt FROM sentiment_themes").collect()[0]['CNT']

if themes_exist > 0:
    themes = session.sql("""
        SELECT TRIM(f.VALUE) as theme, COUNT(*) as mentions
        FROM sentiment_themes,
        LATERAL FLATTEN(input => SPLIT(extracted_themes, ',')) f
        GROUP BY theme
        HAVING mentions >= 2
        ORDER BY mentions DESC
        LIMIT 10
    """).to_pandas()
    
    st.bar_chart(themes.set_index('THEME')['MENTIONS'])
else:
    st.info("Theme extraction in progress...")

# Download
st.markdown("---")
csv = positive.to_csv(index=False)
st.download_button("📥 Download Sentiment Report", data=csv, file_name="sentiment_report.csv", mime="text/csv")

---

## ✅ Tutorial Complete!

### What You've Learned

1. ✅ Used **AI_SENTIMENT()** for text analysis
2. ✅ Analyzed sentiment from patient reviews and social media
3. ✅ Tracked sentiment trends over time
4. ✅ Extracted themes with **CORTEX.COMPLETE()**
5. ✅ Built sentiment dashboards for brand monitoring
6. ✅ Generated actionable insights from unstructured text

### Key Cortex Functions

- **`AI_SENTIMENT(text)`**
  - Returns: -1.0 (negative) to +1.0 (positive)
  - Handles medical terminology and negations
  - Works on any text column

- **`AI_COMPLETE(model, prompt)`**
  - LLM-powered text generation
  - Extract themes, categorize feedback
  - Models: mistral-large2, llama3, etc.

### Sentiment Insights

- **Altoly Positive (0.5-1.0)**: Efficacy, convenience, weight loss
- **Neutral (0.1 to -0.1)**: Mixed experiences, adjusting to side effects
- **Negative (<-0.5)**: GI issues, cost/access barriers, adverse events

### Business Applications

- **Brand monitoring**: Track perception changes
- **Product launches**: Measure initial response
- **Competitive intelligence**: Compare sentiment vs. competitors
- **Customer service**: Prioritize urgent negative feedback
- **Content optimization**: Understand what resonates

### Next Steps

- Add competitor product comparisons
- Integrate real-time social media feeds
- Build alerts for sudden sentiment drops
- Analyze sentiment by patient demographics
- Create automated sentiment reports

### Resources

- [Cortex SENTIMENT](https://docs.snowflake.com/en/user-guide/snowflake-cortex/llm-functions#label-cortex-llm-sentiment)
- [Cortex COMPLETE](https://docs.snowflake.com/en/user-guide/snowflake-cortex/llm-functions#label-cortex-llm-complete)
- [Cortex LLM Functions](https://docs.snowflake.com/en/user-guide/snowflake-cortex/llm-functions)

---

**Congratulations!** You've built a complete sentiment analysis system using Cortex AI.

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `SENTIMENT_DB` database (and all tables within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

⚠️ **Warning**: This action cannot be undone. All data will be permanently deleted.


In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS SENTIMENT_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;